# 01b — Data preprocessing and YOLO11 baseline

The dataset exploration is in `01a_EDA.ipynb`; this notebook loads the annotations again and builds the video-level splits, YOLO labels and baseline experiments.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

DATA_ROOT = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

print("Dataset root:", DATA_ROOT)
print("Dataset exists:", DATA_ROOT.exists())

print("\nDataset contents:")
for p in DATA_ROOT.iterdir():
    print("-", p.name)

print("\nChecking annotations:")
print(
    "Train:",
    (DATA_ROOT / "train/annotations_coco.json").exists()
)
print(
    "Validation:",
    (DATA_ROOT / "val/annotations_coco.json").exists()
)

Dataset root: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
Dataset exists: True

Dataset contents:
- val
- test_images.csv
- test
- train

Checking annotations:
Train: True
Validation: True


In [2]:
with open(DATA_ROOT / "train/annotations_coco.json") as f:
    train_coco = json.load(f)

with open(DATA_ROOT / "val/annotations_coco.json") as f:
    val_coco = json.load(f)

print("TRAIN")
print("Images:", len(train_coco["images"]))
print("Annotations:", len(train_coco["annotations"]))
print("Categories:", train_coco["categories"])

print("\nVALIDATION")
print("Images:", len(val_coco["images"]))
print("Annotations:", len(val_coco["annotations"]))

TRAIN
Images: 24201
Annotations: 1616260
Categories: [{'id': 1, 'name': 'person'}, {'id': 2, 'name': 'bicycle'}, {'id': 3, 'name': 'car'}, {'id': 4, 'name': 'van'}, {'id': 5, 'name': 'truck'}, {'id': 6, 'name': 'tricycle'}, {'id': 7, 'name': 'bus'}, {'id': 8, 'name': 'motor'}]

VALIDATION
Images: 2846
Annotations: 146092


In [3]:
CLASSES = [
    "person", "bicycle", "car", "van",
    "truck", "tricycle", "bus", "motor"
]

def coco_to_dataframe(coco, split):
    images = {img["id"]: img for img in coco["images"]}
    rows = []

    for ann in coco["annotations"]:
        img = images[ann["image_id"]]
        x, y, w, h = ann["bbox"]

        rows.append({
            "split": split,
            "image_id": ann["image_id"],
            "file_name": img["file_name"],
            "clip": img.get("video", img["file_name"].split("/")[0]),
            "class_id": ann["category_id"] - 1,
            "iscrowd": ann.get("iscrowd", 0),
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "area": w * h,
            "image_width": img["width"],
            "image_height": img["height"],
        })

    return pd.DataFrame(rows)

train_df = coco_to_dataframe(train_coco, "train")
val_df = coco_to_dataframe(val_coco, "val")

df = pd.concat([train_df, val_df], ignore_index=True)

# Для статистики класів виключаємо ignored regions
objects = df[(df["iscrowd"] == 0) &
             (df["class_id"].between(0, 7))].copy()

objects["class_name"] = objects["class_id"].map(
    dict(enumerate(CLASSES))
)

print("Total valid objects:", len(objects))
print("Ignored annotations:", (df["iscrowd"] == 1).sum())

display(objects.head())

Total valid objects: 1219648
Ignored annotations: 542704


,split,image_id,file_name,clip,class_id,iscrowd,x,y,width,height,area,image_width,image_height,class_name
0,train,1,images/uav0000013_00000_v/0000001.jpg,uav0000013_00000_v,0,0,424,543,35,88,3080,1344,756,person
1,train,1,images/uav0000013_00000_v/0000001.jpg,uav0000013_00000_v,0,0,416,530,36,85,3060,1344,756,person
2,train,1,images/uav0000013_00000_v/0000001.jpg,uav0000013_00000_v,5,0,531,641,87,114,9918,1344,756,tricycle
3,train,1,images/uav0000013_00000_v/0000001.jpg,uav0000013_00000_v,0,0,591,422,32,79,2528,1344,756,person
4,train,1,images/uav0000013_00000_v/0000001.jpg,uav0000013_00000_v,0,0,454,413,21,48,1008,1344,756,person




# Data Preprocessing

## Frame Sampling and Cross-Validation

A sampling step of 6 reduces the redundancy between consecutive frames.

Three-fold GroupKFold cross-validation separates complete video sequences. The original validation set remains an independent holdout set.



In [9]:

from sklearn.model_selection import GroupKFold

SEED = 42
FRAME_STEP = 6
N_FOLDS = 3
BATCH_SIZE = 2
EPOCHS = 8

# Build frame metadata directly from COCO annotations
def build_frame_table(coco, split):
    rows = []

    for img in coco["images"]:
        file_name = img["file_name"]

        # Example: images/uav0000013_00000_v/0000001.jpg
        clip = Path(file_name).parent.name

        rows.append({
            "split": split,
            "image_id": img["id"],
            "file_name": file_name,
            "clip": clip,
            "frame_id": img.get("frame_id"),
            "width": img["width"],
            "height": img["height"]
        })

    return pd.DataFrame(rows)


train_frames = build_frame_table(train_coco, "train")
holdout_frames = build_frame_table(val_coco, "val")

# Sort frames chronologically within each video
def sample_frames(frame_df, step=6):
    df = frame_df.copy()

    # Filenames are zero-padded in VisDrone
    df = df.sort_values(
        ["clip", "file_name"]
    ).reset_index(drop=True)

    sampled = (
        df.groupby("clip", group_keys=False)
        .head(0)
    )

    sampled = df.groupby(
        "clip", group_keys=False
    ).apply(
        lambda group: group.iloc[::step],
        include_groups=False
    )

    return sampled.reset_index(drop=True)


sampled_train = sample_frames(
    train_frames, FRAME_STEP
)

# Keep clip as an explicit column (independent of
# pandas groupby behavior across versions).
# Reconstruct the clip from each file path.
sampled_train["clip"] = sampled_train[
    "file_name"
].apply(lambda x: Path(x).parent.name)

print("Original train frames:", len(train_frames))
print("Sampled train frames:", len(sampled_train))
print("Original validation frames:", len(holdout_frames))

print("Train clips:", sampled_train["clip"].nunique())
print("Holdout clips:", holdout_frames["clip"].nunique())

display(sampled_train.head())


Original train frames: 24201
Sampled train frames: 4062
Original validation frames: 2846
Train clips: 56
Holdout clips: 7


,split,image_id,file_name,frame_id,width,height,clip
0,train,1,images/uav0000013_00000_v/0000001.jpg,1,1344,756,uav0000013_00000_v
1,train,7,images/uav0000013_00000_v/0000007.jpg,7,1344,756,uav0000013_00000_v
2,train,13,images/uav0000013_00000_v/0000013.jpg,13,1344,756,uav0000013_00000_v
3,train,19,images/uav0000013_00000_v/0000019.jpg,19,1344,756,uav0000013_00000_v
4,train,25,images/uav0000013_00000_v/0000025.jpg,25,1344,756,uav0000013_00000_v


In [10]:

gkf = GroupKFold(n_splits=N_FOLDS)

sampled_train["fold"] = -1

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(
        sampled_train,
        groups=sampled_train["clip"]
    )
):
    sampled_train.loc[val_idx, "fold"] = fold

    train_part = sampled_train.iloc[train_idx]
    val_part = sampled_train.iloc[val_idx]

    train_clips_fold = set(train_part["clip"])
    val_clips_fold = set(val_part["clip"])

    assert not (
        train_clips_fold & val_clips_fold
    ), "Data leakage detected!"

    print(f"\nFold {fold + 1}")
    print("Train frames:", len(train_idx))
    print("Val frames:", len(val_idx))
    print("Train clips:", len(train_clips_fold))
    print("Val clips:", len(val_clips_fold))

# Verify each sampled frame belongs to exactly one fold
assert sampled_train["fold"].between(
    0, N_FOLDS - 1
).all()

# Verify the original holdout set is independent
assert not (
    set(sampled_train["clip"]) &
    set(holdout_frames["clip"])
)

print("\nCross-validation splits verified!")
display(sampled_train["fold"].value_counts().sort_index())



Fold 1
Train frames: 2705
Val frames: 1357
Train clips: 37
Val clips: 19

Fold 2
Train frames: 2706
Val frames: 1356
Train clips: 37
Val clips: 19

Fold 3
Train frames: 2713
Val frames: 1349
Train clips: 38
Val clips: 18

Cross-validation splits verified!


fold
0    1357
1    1356
2    1349
Name: count, dtype: int64

In [11]:

import json

OUTPUT_DIR = Path("/kaggle/working/preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sampled_train.to_csv(
    OUTPUT_DIR / "train_3fold_step6.csv",
    index=False
)

holdout_frames.to_csv(
    OUTPUT_DIR / "holdout_val.csv",
    index=False
)

config = {
    "seed": SEED,
    "frame_step": FRAME_STEP,
    "n_folds": N_FOLDS,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "split_method": "GroupKFold",
    "group_column": "clip",
    "holdout": "original_val",
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Preprocessing files saved:")
for path in OUTPUT_DIR.iterdir():
    print(path.name, "-", path.stat().st_size, "bytes")


Preprocessing files saved:
holdout_val.csv - 225042 bytes
config.json - 188 bytes
train_3fold_step6.csv - 341414 bytes



## YOLO Label Validation

The original YOLO annotations are checked for valid class IDs, normalized bounding box coordinates, and consistency with the COCO annotations.

This ensures that the labels can be loaded correctly before model training.


In [12]:

from pathlib import Path
import random
import numpy as np

rng = random.Random(SEED)

# Inspect 30 sampled training frames
check_samples = sampled_train.sample(
    n=min(30, len(sampled_train)),
    random_state=SEED
)

# Count target annotations in COCO
coco_counts = (
    train_df[
        (train_df["iscrowd"] == 0) &
        (train_df["class_id"].between(0, 7))
    ]
    .groupby("image_id")
    .size()
)

errors = []
checked_boxes = 0

for row in check_samples.itertuples():
    image_path = DATA_ROOT / "train" / row.file_name

    label_path = (
        DATA_ROOT / "train" / "labels" /
        Path(row.file_name).parent.name /
        (Path(row.file_name).stem + ".txt")
    )

    if not image_path.exists():
        errors.append(f"Missing image: {image_path}")
        continue

    if not label_path.exists():
        errors.append(f"Missing label: {label_path}")
        continue

    lines = [
        line.strip()
        for line in label_path.read_text().splitlines()
        if line.strip()
    ]

    expected = int(coco_counts.get(row.image_id, 0))

    if len(lines) != expected:
        errors.append(
            f"{row.file_name}: YOLO={len(lines)}, COCO={expected}"
        )

    for line in lines:
        values = line.split()

        if len(values) != 5:
            errors.append(f"Invalid label: {line}")
            continue

        cls = int(values[0])
        cx, cy, w, h = map(float, values[1:])

        if not (
            0 <= cls < 8 and
            0 <= cx <= 1 and
            0 <= cy <= 1 and
            0 < w <= 1 and
            0 < h <= 1
        ):
            errors.append(f"Invalid values: {line}")

        checked_boxes += 1

print("Checked frames:", len(check_samples))
print("Checked bounding boxes:", checked_boxes)
print("Errors:", len(errors))

for error in errors[:10]:
    print(error)

if errors:
    raise ValueError("YOLO label validation failed.")

print("YOLO label validation passed!")


Checked frames: 30
Checked bounding boxes: 1454
Errors: 0
YOLO label validation passed!



## YOLO Cross-Validation Configuration

Three fold-specific configurations are generated using the same sequence-level splits.

Each configuration contains the training and validation image lists for its corresponding fold. The original image and annotation files are reused without duplicating the dataset.


In [13]:

import yaml

YOLO_DIR = Path("/kaggle/working/yolo_cv")
YOLO_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    i: name for i, name in enumerate(CLASSES)
}

def get_image_path(file_name):
    return str(DATA_ROOT / "train" / file_name)

for fold in range(N_FOLDS):

    fold_dir = YOLO_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_part = sampled_train[
        sampled_train["fold"] != fold
    ].copy()

    val_part = sampled_train[
        sampled_train["fold"] == fold
    ].copy()

    # Ensure no video sequence appears in both sets
    assert set(train_part["clip"]).isdisjoint(
        set(val_part["clip"])
    )

    train_paths = train_part["file_name"].apply(
        get_image_path
    ).tolist()

    val_paths = val_part["file_name"].apply(
        get_image_path
    ).tolist()

    train_txt = fold_dir / "train.txt"
    val_txt = fold_dir / "val.txt"

    train_txt.write_text(
        "\n".join(train_paths) + "\n"
    )

    val_txt.write_text(
        "\n".join(val_paths) + "\n"
    )

    config = {
        "train": str(train_txt),
        "val": str(val_txt),
        "nc": 8,
        "names": CLASS_NAMES
    }

    yaml_path = fold_dir / "data.yaml"

    with open(yaml_path, "w") as f:
        yaml.safe_dump(
            config,
            f,
            sort_keys=False
        )

    print(f"\nFold {fold + 1}")
    print("Training images:", len(train_paths))
    print("Validation images:", len(val_paths))
    print("Config:", yaml_path)



Fold 1
Training images: 2705
Validation images: 1357
Config: /kaggle/working/yolo_cv/fold_0/data.yaml

Fold 2
Training images: 2706
Validation images: 1356
Config: /kaggle/working/yolo_cv/fold_1/data.yaml

Fold 3
Training images: 2713
Validation images: 1349
Config: /kaggle/working/yolo_cv/fold_2/data.yaml


In [14]:

for fold in range(N_FOLDS):
    yaml_path = YOLO_DIR / f"fold_{fold}/data.yaml"

    with open(yaml_path) as f:
        config = yaml.safe_load(f)

    train_paths = Path(config["train"]).read_text().splitlines()
    val_paths = Path(config["val"]).read_text().splitlines()

    assert len(train_paths) > 0
    assert len(val_paths) > 0

    # Check paths and corresponding YOLO labels
    for image_str in train_paths[:5] + val_paths[:5]:
        image_path = Path(image_str)

        label_path = Path(
            str(image_path).replace(
                "/images/", "/labels/"
            )
        ).with_suffix(".txt")

        assert image_path.exists(), image_path
        assert label_path.exists(), label_path

    assert set(train_paths).isdisjoint(set(val_paths))

    print(
        f"Fold {fold + 1}: OK | "
        f"train={len(train_paths)} | "
        f"val={len(val_paths)}"
    )

print("\nAll YOLO configurations verified!")


Fold 1: OK | train=2705 | val=1357
Fold 2: OK | train=2706 | val=1356
Fold 3: OK | train=2713 | val=1349

All YOLO configurations verified!



# Model Training and Evaluation

## YOLO11n Baseline

The first experiment uses a pretrained YOLO11n detector with 3-fold sequence-level cross-validation.

Configuration:
- Frame sampling step: 6
- Batch size: 2
- Epochs: 6
- Input resolution: 640
- Optimizer: AdamW
- Random seed: 42

The same training configuration is applied to all three folds.


In [16]:

import sys
import subprocess
import torch

subprocess.check_call([
    sys.executable, "-m", "pip",
    "install", "-q", "ultralytics"
])

import ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics: 8.4.203
CUDA available: True
GPU count: 2
GPU 0: Tesla T4


In [17]:

from pathlib import Path
import yaml


YOLO_DIR = Path("/kaggle/working/yolo_cv")

for fold in range(3):
    config_path = YOLO_DIR / f"fold_{fold}" / "data.yaml"

    assert config_path.exists(), f"Missing {config_path}"

    with open(config_path) as f:
        config = yaml.safe_load(f)

    for split in ["train", "val"]:
        image_list = Path(config[split])
        assert image_list.exists(), image_list

        paths = image_list.read_text().splitlines()
        missing_images = []
        missing_labels = []

        for image_str in paths:
            image_path = Path(image_str)

            label_path = Path(
                str(image_path).replace(
                    "/images/", "/labels/"
                )
            ).with_suffix(".txt")

            if not image_path.exists():
                missing_images.append(str(image_path))

            if not label_path.exists():
                missing_labels.append(str(label_path))

        print(
            f"Fold {fold + 1} | {split}: "
            f"{len(paths)} images | "
            f"missing images={len(missing_images)} | "
            f"missing labels={len(missing_labels)}"
        )

        assert not missing_images, missing_images[:5]
        assert not missing_labels, missing_labels[:5]

print("\nAll folds are ready for training!")


Fold 1 | train: 2705 images | missing images=0 | missing labels=0
Fold 1 | val: 1357 images | missing images=0 | missing labels=0
Fold 2 | train: 2706 images | missing images=0 | missing labels=0
Fold 2 | val: 1356 images | missing images=0 | missing labels=0
Fold 3 | train: 2713 images | missing images=0 | missing labels=0
Fold 3 | val: 1349 images | missing images=0 | missing labels=0

All folds are ready for training!


In [18]:

import gc
import json
import pandas as pd
import torch
from pathlib import Path
from ultralytics import YOLO

assert torch.cuda.is_available(), "Enable GPU before training!"

MODEL_NAME = "yolo11n.pt"
IMG_SIZE = 640
BATCH_SIZE = 2
EPOCHS = 6
SEED = 42

PROJECT_DIR = Path("/kaggle/working/yolo_baseline")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

cv_results = []

import time

NIGHT_START = time.monotonic()

for fold in range(3):

    print(f"\n{'=' * 55}")
    print(f"TRAINING FOLD {fold + 1}/3")
    print(f"{'=' * 55}")

    data_yaml = (
        Path("/kaggle/working/yolo_cv")
        / f"fold_{fold}"
        / "data.yaml"
    )

    # Start each fold from the same pretrained weights
    model = YOLO(MODEL_NAME)

    model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0,
        workers=2,
        optimizer="AdamW",
        lr0=0.001,
        seed=SEED,
        deterministic=True,
        amp=True,
        patience=EPOCHS,
        save=True,
        save_period=1,
        plots=True,
        cache=False,
        project=str(PROJECT_DIR),
        name=f"fold_{fold}",
        exist_ok=False,
    )

    # Load best checkpoint for evaluation
    run_dir = Path(model.trainer.save_dir)
    best_weights = run_dir / "weights" / "best.pt"

    assert best_weights.exists(), best_weights

    best_model = YOLO(str(best_weights))

    metrics = best_model.val(
        data=str(data_yaml),
        split="val",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0,
        workers=2,
        plots=False,
        project=str(PROJECT_DIR),
        name=f"fold_{fold}_evaluation",
        exist_ok=False,
    )

    row = {
        "fold": fold + 1,
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),
        "checkpoint": str(best_weights),
    }

    cv_results.append(row)

    # Save intermediate metrics after every fold
    results_df = pd.DataFrame(cv_results)
    results_df.to_csv(
        PROJECT_DIR / "cv_results.csv",
        index=False
    )

    print(f"\nFold {fold + 1} results:")
    print(results_df.tail(1).to_string(index=False))

    del model, best_model, metrics
    gc.collect()
    torch.cuda.empty_cache()

print("\nALL FOLDS COMPLETED")

results_df = pd.DataFrame(cv_results)
display(results_df)

print("\nMean metrics:")
print(
    results_df[["mAP50-95", "mAP50", "mAP75"]]
    .agg(["mean", "std"])
)



TRAINING FOLD 1/3
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_0, nbs=64, nms=None, opse

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/6     0.479G      1.921      1.372      1.019         55        640: 100% ━━━━━━━━━━━━ 1353/1353 15.1it/s 1:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 19.7it/s 17.3s
                   all       1357      59035      0.266      0.257      0.196     0.0937

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        3/6     0.479G      1.856      1.265     0.9977         81        640: 100% ━━━━━━━━━━━━ 1353/1353 15.3it/s 1:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 19.9it/s 17.1s
                   all       1357      59035      0.292      0.256      0.202     0.0958

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        4/6     0.479G      1.803      1.173     0.9807         28        640: 100% ━━━━━━━━━━━━ 1353/1353 15.3it/s 1:28
                 Class     Images  Inst

,fold,mAP50-95,mAP50,mAP75,checkpoint
0,1,0.114373,0.227232,0.102711,/kaggle/working/yolo_baseline/fold_0/weights/b...
1,2,0.101959,0.199830,0.092723,/kaggle/working/yolo_baseline/fold_1/weights/b...
2,3,0.101920,0.214637,0.086001,/kaggle/working/yolo_baseline/fold_2/weights/b...



Mean metrics:
      mAP50-95     mAP50     mAP75
mean  0.106084  0.213900  0.093812
std   0.007179  0.013716  0.008408



## YOLO11s — Model Size and Resolution Experiments

Two additional configurations are tested using the same first cross-validation fold:

1. YOLO11s with 640px input resolution and AdamW.
2. YOLO11s with 960px input resolution and AdamW.

Both experiments use the same batch size, number of epochs, seed, and data split. The goal is to compare YOLO11n and YOLO11s and evaluate the influence of input resolution on object detection performance.

The experiments are exploratory; full cross-validation will be required for the selected configuration.


In [19]:

import time
import gc
import json
import torch
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path("/kaggle/working/yolo_experiments")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = str(
    Path("/kaggle/working/yolo_cv/fold_0/data.yaml")
)

assert Path(DATA_YAML).exists()
assert torch.cuda.is_available()

# Reserve time before Kaggle's 12-hour limit
MAX_START_HOURS = 8.0

experiments = [
    {
        "name": "yolo11s_640_adamw",
        "model": "yolo11s.pt",
        "imgsz": 640,
        "optimizer": "AdamW",
        "lr0": 0.001,
    },
    {
        "name": "yolo11s_960_adamw",
        "model": "yolo11s.pt",
        "imgsz": 960,
        "optimizer": "AdamW",
        "lr0": 0.001,
    },
]

experiment_results = []

for exp in experiments:

    elapsed_hours = (
        time.monotonic() - NIGHT_START
    ) / 3600

    if elapsed_hours >= MAX_START_HOURS:
        print(
            "Skipping remaining experiments: "
            "nightly time budget reached."
        )
        break

    print("\n" + "=" * 55)
    print("EXPERIMENT:", exp["name"])
    print("Elapsed hours:", round(elapsed_hours, 2))
    print("=" * 55)

    model = YOLO(exp["model"])

    train_start = time.monotonic()

    model.train(
        data=DATA_YAML,
        epochs=6,
        batch=2,
        imgsz=exp["imgsz"],
        optimizer=exp["optimizer"],
        lr0=exp["lr0"],
        device=0,
        workers=2,
        seed=42,
        deterministic=True,
        amp=True,
        cache=False,
        patience=6,
        save=True,
        save_period=1,
        plots=True,
        project=str(PROJECT_DIR),
        name=exp["name"],
        exist_ok=True,
    )

    train_minutes = (
        time.monotonic() - train_start
    ) / 60

    best_path = (
        Path(model.trainer.save_dir)
        / "weights" / "best.pt"
    )

    assert best_path.exists(), best_path

    best_model = YOLO(str(best_path))

    metrics = best_model.val(
        data=DATA_YAML,
        split="val",
        imgsz=exp["imgsz"],
        batch=2,
        device=0,
        workers=2,
        plots=False,
        project=str(PROJECT_DIR),
        name=exp["name"] + "_eval",
        exist_ok=True,
    )

    result = {
        "experiment": exp["name"],
        "fold": 0,
        "model": exp["model"],
        "imgsz": exp["imgsz"],
        "optimizer": exp["optimizer"],
        "epochs": 6,
        "batch": 2,
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),
        "train_minutes": round(train_minutes, 2),
        "weights": str(best_path),
    }

    experiment_results.append(result)

    pd.DataFrame(experiment_results).to_csv(
        PROJECT_DIR / "comparison.csv",
        index=False
    )

    print("\nCompleted:", exp["name"])
    print("mAP50-95:", result["mAP50-95"])
    print("Training time:", train_minutes, "minutes")

    del model, best_model, metrics
    gc.collect()
    torch.cuda.empty_cache()

print("\nNIGHT EXPERIMENTS FINISHED")

if experiment_results:
    display(pd.DataFrame(experiment_results))
else:
    print("No additional experiment was started.")



EXPERIMENT: yolo11s_640_adamw
Elapsed hours: 0.59
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

,experiment,fold,model,imgsz,optimizer,epochs,batch,mAP50-95,mAP50,mAP75,train_minutes,weights
0,yolo11s_640_adamw,0,yolo11s.pt,640,AdamW,6,2,0.139723,0.275463,0.125648,12.3,/kaggle/working/yolo_experiments/yolo11s_640_a...
1,yolo11s_960_adamw,0,yolo11s.pt,960,AdamW,6,2,0.178977,0.345127,0.163915,14.2,/kaggle/working/yolo_experiments/yolo11s_960_a...
